# 3D multi-patch assembly — refinement propagation across shared faces

Demonstrates the newly added 3D support in `PatchAssembly.detect_interfaces()` /
`refine_with_propagation()`: a 3D facet is a **face** (two varying directions), not
an edge (one varying direction) like in 2D, so two matching faces can correspond in
up to 8 ways — the symmetry group of a rectangle (axis transpose x independent
per-axis reversal) instead of just "same order"/"reversed". See
`tests/future/test_assembly_3d.py` for the exhaustive correctness tests; this
notebook is about the *workflow* and a visual of the propagation process.

**Geometry**: a "plus" (cross) of 5 unit-cube `BSplineVolume` patches — one **center**
cube and 4 neighbors (**east**, **west**, **north**, **south**) sharing a face with
it. This is deliberately not just a 2-patch toy case: it lets one
`refine_with_propagation()` call hit **multiple neighbors at once** whenever they
share the same varying direction (here: refining the center along $z$ propagates to
*all four* neighbors in a single call, since $z$ is a varying direction of every one
of the 4 shared faces).

**Important limitation** (unchanged from the 2D implementation): propagation is only
**one hop** — it reaches direct neighbors of the patch being refined, not neighbors
of neighbors. That is not a limitation here (every neighbor touches the center
directly), but it would matter for a longer chain of patches.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (
    BSpline, BSplineVolume, Patch, ControlPointManager, PatchAssembly,
    SubdivisionRefiner, GlobalDOFManager, PatchDOFManager, IGABasis1D,
    Material, Elasticity3D, PatchIntegrator,
)

## Geometry — a "plus" of 5 unit cubes

Each cube is a minimal (degree 1, 1 element/direction) `BSplineVolume`, 8 control
points at its corners. Shared corners between adjacent cubes reuse the same
`ControlPointManager` entry (the same `add()` memoization pattern used throughout
this codebase's multi-patch tests) so they already carry the same global id *before*
any detection step runs.

In [ ]:
mgr = ControlPointManager(dim=3)
_cache = {}

def add(x, y, z):
    key = (round(x, 6), round(y, 6), round(z, 6))
    if key not in _cache:
        _cache[key] = mgr.add_point([x, y, z])
    return _cache[key]

def build_cube_ids(x0, y0, z0):
    """8 corner ids of a unit cube at (x0,y0,z0)-(x0+1,y0+1,z0+1), u-fastest."""
    ids = []
    for iw in range(2):
        for iv in range(2):
            for iu in range(2):
                ids.append(add(x0 + iu, y0 + iv, z0 + iw))
    return ids

# Planar cross: center + 4 neighbors, all touching the center on a full face.
positions = {
    'center': (0, 0, 0),
    'east':   (1, 0, 0),
    'west':   (-1, 0, 0),
    'north':  (0, 1, 0),
    'south':  (0, -1, 0),
}
order = list(positions.keys())

kv = np.array([0., 0., 1., 1.])

def make_patch(ids):
    tensor = BSplineVolume(BSpline(1, kv), BSpline(1, kv), BSpline(1, kv))
    return Patch(tensor, mgr, ids, [2, 2, 2])

patches = {name: make_patch(build_cube_ids(*positions[name])) for name in order}
print(f'{len(patches)} patches, {mgr.n_points} shared control points')

## Visualization helper

No dedicated 3D-solid plotting helper exists yet in `postprocessing/` (unlike
`plot_patches_2d()` for 2D patches) — a small matplotlib wireframe helper is enough
here: draw every iso-parametric line of the control net (not just the 12
bounding-box edges), so mesh refinement is visible as "more lines", not just a
bigger box.

In [ ]:
def plot_patch_wireframe(patch, ax, color, linewidth=1.0, alpha=0.9, label=None):
    nu, nv, nw = patch.local_shape
    pts = np.array([patch.control_point(iu + iv*nu + iw*nu*nv)
                     for iw in range(nw) for iv in range(nv) for iu in range(nu)])
    pts = pts.reshape(nw, nv, nu, 3).transpose(2, 1, 0, 3)  # -> [iu, iv, iw, xyz]

    for iv in range(nv):
        for iw in range(nw):
            ax.plot(*pts[:, iv, iw, :].T, color=color, linewidth=linewidth, alpha=alpha)
    for iu in range(nu):
        for iw in range(nw):
            ax.plot(*pts[iu, :, iw, :].T, color=color, linewidth=linewidth, alpha=alpha)
    for iu in range(nu):
        for iv in range(nv):
            ax.plot(*pts[iu, iv, :, :].T, color=color, linewidth=linewidth, alpha=alpha)

    flat = pts.reshape(-1, 3)
    ax.scatter(*flat.T, color=color, s=8, alpha=alpha)
    if label:
        ax.text(*flat.mean(axis=0), label, color=color, fontsize=10, weight='bold')


def plot_assembly(patches, order, title):
    colors = plt.cm.tab10(np.arange(len(order)))
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(projection='3d')
    for name, color in zip(order, colors):
        plot_patch_wireframe(patches[name], ax, color, label=name)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_box_aspect([3, 3, 1])
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


plot_assembly(patches, order, 'Initial assembly (1 element/direction per cube)')

## Detecting shared control points and interfaces

`detect_shared_control_points()` finds control point ids common to several patches
(plain set intersection, already dimension-generic). `detect_interfaces()` then
matches *facets* between patches: for a 3D pair this now checks all 8 possible face
correspondences (new code added for this work) instead of throwing as it used to.

In [ ]:
assembly = PatchAssembly()
for name in order:
    assembly.add_patch(patches[name])

assembly.detect_shared_control_points()
assembly.detect_interfaces()

itfs = assembly.get_interfaces()
print(f'{len(itfs)} interfaces detected\n')
for itf in itfs:
    a, b = order[itf['patch_a']], order[itf['patch_b']]
    print(f"{a:7s} <-> {b:7s}  "
          f"facet_a=(dir={itf['direction_a']}, side={itf['side_a']})  "
          f"facet_b=(dir={itf['direction_b']}, side={itf['side_b']})  "
          f"varying_directions_a={itf['varying_directions_a']}  "
          f"transpose={itf['transpose']}")

## Refinement with propagation — one call, four neighbors

Every one of the 4 shared faces has $z$ (direction 2) as one of its two varying
directions (the center-east/west faces vary over $(y,z)$, the center-north/south
faces vary over $(x,z)$). Refining the **center** patch along $z$ therefore
propagates to **all four neighbors in a single `refine_with_propagation()` call** —
the function loops over every detected interface, not just the first match.

In [ ]:
def refine_1d_fn(patch, direction, protected_ids):
    SubdivisionRefiner(direction=direction, n_levels=2).refine_1d(patch, protected_ids)

center_idx = order.index('center')

print('Before:', {name: patches[name].local_shape for name in order})
assembly.refine_with_propagation(patch_index=center_idx, direction=2, refine_1d_fn=refine_1d_fn)
print('After refining center along z (propagated to all 4 neighbors):')
print('           ', {name: patches[name].local_shape for name in order})

plot_assembly(patches, order, 'After propagating refinement along z (one call, 4 neighbors)')

## Finishing the refinement — x and y

The $x$ direction is a varying direction of the north/south faces only (their shared
face varies over $(x,z)$); the $y$ direction is a varying direction of the east/west
faces only (varying over $(y,z)$). Two more calls finish refining every shared face
in every direction.

In [ ]:
assembly.refine_with_propagation(patch_index=center_idx, direction=0, refine_1d_fn=refine_1d_fn)
assembly.refine_with_propagation(patch_index=center_idx, direction=1, refine_1d_fn=refine_1d_fn)

print({name: (patches[name].local_shape, patches[name].n_cp) for name in order})

plot_assembly(patches, order, 'Fully refined assembly (all shared faces grown on both sides)')

## Integrity check — shared faces still coincide exactly

After independent refinement on each side followed by propagation's merge step, the
shared face control points must carry the **same global ids** (not just
coordinate-identical duplicates) on both patches.

In [ ]:
def check_shared_face(name_a, dir_a, side_a, name_b, dir_b, side_b):
    pa, pb = patches[name_a], patches[name_b]
    face_a = pa.boundary_control_points(dir_a, side_a)
    face_b = pb.boundary_control_points(dir_b, side_b)
    ids_a = {pa.global_indices[i] for i in face_a}
    ids_b = {pb.global_indices[i] for i in face_b}
    ok = (ids_a == ids_b) and (len(face_a) == len(face_b))
    print(f'{name_a:7s} <-> {name_b:7s}: {len(face_a)} CPs, ids match: {ok}')
    return ok

all_ok = True
all_ok &= check_shared_face('center', 0, 1, 'east', 0, 0)
all_ok &= check_shared_face('center', 0, 0, 'west', 0, 1)
all_ok &= check_shared_face('center', 1, 1, 'north', 1, 0)
all_ok &= check_shared_face('center', 1, 0, 'south', 1, 1)
print(f'\nAll shared faces merged correctly: {all_ok}')

## Bonus: the assembly is usable for real computation

Beyond topology: assign a shared `PatchDOFManager` to every patch, update it after
propagation so merged boundary control points get the same global dofs
(`update_dof_managers()`), then assemble each patch's stiffness contribution and
scatter it into one global matrix — symmetric, as it must be, confirming the merged
assembly is numerically consistent, not just visually aligned.

In [ ]:
gdm = GlobalDOFManager([3] * mgr.n_points)
for name in order:
    pdm = PatchDOFManager(3, patches[name].global_indices, gdm)
    patches[name] = Patch(patches[name].tensor, patches[name].cp_manager,
                           list(patches[name].global_indices),
                           list(patches[name].local_shape), pdm)

assembly.update_dof_managers(gdm, dofs_per_cp=3)

law = Elasticity3D(Material(E=210000.0, nu=0.3))
n_dof = gdm.n_control_points() * 3
K = np.zeros((n_dof, n_dof))

for name in order:
    patch = patches[name]
    su, sv, sw = patch.tensor.components
    basis_u = IGABasis1D.build(su, su.degree + 1)
    basis_v = IGABasis1D.build(sv, sv.degree + 1)
    basis_w = IGABasis1D.build(sw, sw.degree + 1)
    integrator = PatchIntegrator(patch, basis_u, basis_v, basis_w, law)
    K_local = integrator.integrate_stiffness().toarray()

    for a in range(patch.n_cp):
        dofs_a = patch.dof_manager.get_global_dof_indices(a)
        for b in range(patch.n_cp):
            dofs_b = patch.dof_manager.get_global_dof_indices(b)
            for ia, ga in enumerate(dofs_a):
                for ib, gb in enumerate(dofs_b):
                    K[ga, gb] += K_local[3*a + ia, 3*b + ib]

print(f'Global K: {K.shape}, symmetry error = {np.max(np.abs(K - K.T)):.2e}')